# Classes in Python

This module introduces Python's object-oriented features. Readers are assumed to be familiar with general OOP concepts (classes, instances, inheritance) from other languages; the focus here is on how Python realizes them and where it differs.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. Python's place in the OOP landscape
2. Everything is an object
3. Defining a class and creating instances
4. `self` and bound methods
5. `__init__` is not the constructor
6. Instance attributes as a namespace
7. Class attributes vs. instance attributes
8. Methods are functions stored on the class
9. Encapsulation by convention
10. Properties for controlled access
11. Inheritance
12. `super()` and cooperative initialization
13. Method overriding and polymorphism
14. `isinstance` and `issubclass`
15. Composition vs. inheritance
16. Duck typing
17. Abstract base classes
18. Special methods (dunders)
19. `__str__` vs. `__repr__`
20. Equality and hashing
21. Custom exceptions through inheritance
22. Dataclasses
23. `classmethod` and `staticmethod`
24. Classes are themselves objects
25. Real-world design principles
26. Common mistakes

---

## 1. Python's place in the OOP landscape

Python is a multi-paradigm language. Classes were not part of the original 1991 release in the form they have today; the modern class system (so-called "new-style classes") was introduced in Python 2.2 and became the only style in Python 3. The design draws primarily from Modula-3, with influences from C++ and Lisp, but deliberately not from Smalltalk's pure-OOP model. As a consequence, Python's object system feels lighter than that of Java or C#: there are no required access modifiers, no required class-per-file convention, and no requirement that code live inside a class at all. Whole programs can be written without defining a single class.

Two design choices are worth stating up front, because they explain much of what follows. First, Python trusts the programmer: features like access control are conventions rather than enforced rules. Second, Python prefers explicitness in places where other languages hide machinery — the instance is passed explicitly to methods as `self`, and class behavior is implemented through visible special methods rather than compiler magic.

## 2. Everything is an object

In Python, every value is an object: integers, strings, functions, modules, exceptions, and classes themselves. Each object has a type, a unique identity, and a set of attributes.

In [ ]:
type(42)            # <class 'int'>
type("hello")       # <class 'str'>
type([1, 2, 3])     # <class 'list'>
type(print)         # <class 'builtin_function_or_method'>
type(int)           # <class 'type'>

id(42)              # unique integer identity
42 is 42            # True (identity comparison)

Because functions and classes are themselves objects, they can be assigned to variables, stored in containers, and passed as arguments. This property — being usable wherever any other value can be used — is what is meant by "first-class objects."

In [ ]:
operations = [str.upper, str.lower, str.title]
for op in operations:
    print(op("hello world"))

The class system described in the rest of this document is built on this same principle: a class is an object, an instance is an object, and the relationship between them is itself expressed through ordinary attribute lookup.

## 3. Defining a class and creating instances

A class is defined with the `class` keyword. Instances are created by calling the class as if it were a function.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


account = Account("Anna Smith", balance=500.0)
account.deposit(150.0)
account.withdraw(80.0)
print(account.balance)   # 570.0

`Account` is the class — the description of a kind of thing. `account` is an instance — a concrete object of that kind. The class describes which attributes instances will have (`owner`, `balance`) and which operations are defined on them (`deposit`, `withdraw`).

## 4. `self` and bound methods

Inside a method, `self` refers to the instance on which the method is being called. It is passed implicitly when using the `instance.method(...)` syntax, but the call can be written explicitly through the class:

In [ ]:
account = Account("Anna Smith", 500.0)

account.deposit(150.0)             # implicit form
Account.deposit(account, 150.0)    # explicit form, identical effect

Python chooses to expose `self` in the method signature; languages such as Java and C++ keep the equivalent (`this`) implicit. Treating `self` as an ordinary parameter is what makes `Account.deposit(account, 150.0)` work, and it is also why methods defined inside a class always declare `self` as their first parameter.

Method access produces different objects depending on whether it goes through the class or an instance:

In [ ]:
Account.deposit         # <function Account.deposit at 0x...>
account.deposit         # <bound method Account.deposit of <Account ...>>

A bound method is a callable that already has the instance attached. Invoking it does not require passing the instance again.

## 5. `__init__` is not the constructor

`__init__` is often called "the constructor," but strictly speaking it is the _initializer_. By the time `__init__` runs, the instance already exists; `__init__` only assigns attributes to it. The actual construction step is performed by `__new__`, which is rarely overridden outside of immutable types or metaclass code.

In [ ]:
class Account:
    def __new__(cls, *args, **kwargs):
        instance = super().__new__(cls)
        # the instance exists at this point, before __init__ runs
        return instance

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

For ordinary classes, `__new__` does not need to be defined. The distinction matters when subclassing immutable built-in types (such as `tuple` or `str`), where attribute assignment in `__init__` would be too late.

## 6. Instance attributes as a namespace

Each instance carries its attributes in a dictionary accessible as `__dict__`. Attribute access — `account.balance` — is implemented as a lookup in this dictionary (with class-level fallback, described next).

In [ ]:
account = Account("Anna Smith", 500.0)
account.__dict__
# {'owner': 'Anna Smith', 'balance': 500.0}

account.email = "anna@example.com"   # added at runtime
del account.email                    # removed at runtime

This flexibility has a cost. Typos do not raise errors; they create new attributes silently:

In [ ]:
account.balnace = 100   # creates a new attribute, does not modify balance

For classes where this matters, `__slots__` can restrict the allowed attribute names and reduce per-instance memory overhead:

In [ ]:
class Account:
    __slots__ = ("owner", "balance")

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance


account = Account("Anna Smith", 500.0)
account.email = "anna@example.com"   # AttributeError

`__slots__` is an optimization, not a default. Most classes do without it.

## 7. Class attributes vs. instance attributes

Attributes can be defined on the class itself, in which case they are shared across all instances, or on individual instances. When an attribute is read, Python first checks the instance's `__dict__`, then the class, then the class's parent classes in order.

In [ ]:
class Account:
    currency: str = "EUR"           # class attribute, shared
    minimum_balance: float = 0.0    # class attribute, shared

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner          # instance attribute
        self.balance = balance      # instance attribute


a = Account("Anna Smith", 500.0)
b = Account("Ben Jones", 1000.0)

a.currency                  # "EUR" — read from class
Account.currency = "USD"
b.currency                  # "USD" — class attribute changed for all
a.currency = "GBP"          # creates instance attribute, shadows class
b.currency                  # still "USD"

A common pitfall is using a mutable object as a class attribute:

In [ ]:
class Account:
    transactions = []   # shared across all instances — usually a bug

    def deposit(self, amount: float) -> None:
        self.transactions.append(amount)

Every `Account` instance ends up sharing the same list. The standard fix is to assign the mutable object inside `__init__`, so each instance gets its own:

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.transactions: list[float] = []

## 8. Methods are functions stored on the class

A method is an ordinary function that happens to be defined inside a class body. Python's machinery for turning it into a bound method on access is the only thing that distinguishes it from a free-standing function.

In [ ]:
class Account:
    def deposit(self, amount: float) -> None:
        self.balance += amount


Account.deposit              # <function Account.deposit at 0x...>
account.deposit              # <bound method Account.deposit of ...>

Because methods are functions stored as class attributes, they can be added or replaced after the class has been defined:

In [ ]:
def close(self) -> None:
    self.balance = 0.0
    self.closed = True


Account.close = close
account.close()

This is uncommon in production code but illustrates that classes are mutable runtime objects, not compile-time declarations.

## 9. Encapsulation by convention

Python provides no `private` keyword. Three conventions are used instead:

- `name` — public; part of the class's intended interface.
- `_name` — internal; not part of the public API. External code is asked, not forced, to leave it alone.
- `__name` — name-mangled. The attribute is rewritten by the interpreter to `_ClassName__name` to avoid accidental clashes in subclasses. This is not access control.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self._balance = balance     # internal by convention
        self.__pin = "0000"         # name-mangled to _Account__pin


account = Account("Anna Smith", 500.0)
account._balance              # accessible; the underscore is only a hint
account.__pin                 # AttributeError
account._Account__pin         # works — mangling is not security

The underlying philosophy is that misuse of internal attributes is the caller's responsibility, not something the language should prevent. In practice, library authors document which names are public and rely on the leading underscore to mark the rest.

## 10. Properties for controlled access

When read or write access needs to run code (validation, lazy computation, format conversion), `@property` allows a method to be called using attribute syntax. This means a class can start with a plain attribute and migrate to a property later without changing any calling code.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self._balance = value

    @property
    def is_overdrawn(self) -> bool:
        return self._balance < 0


account = Account("Anna Smith", 500.0)
account.balance               # 500.0 — looks like attribute access
account.balance = 700.0       # setter runs, validation enforced
account.balance = -50.0       # ValueError
account.is_overdrawn          # computed on each access

Read-only properties (no setter) are common for derived values. Writeable properties are typically used to enforce invariants on stored fields.

## 11. Inheritance

A class can extend another, inheriting all of its attributes and methods. The base class is named in parentheses after the class name.

In [ ]:
class SavingsAccount(Account):
    def __init__(
        self,
        owner: str,
        balance: float = 0.0,
        interest_rate: float = 0.02,
    ) -> None:
        super().__init__(owner, balance)
        self.interest_rate = interest_rate

    def apply_interest(self) -> None:
        self.balance += self.balance * self.interest_rate


savings = SavingsAccount("Anna Smith", 1000.0, interest_rate=0.03)
savings.deposit(200.0)        # inherited from Account
savings.apply_interest()      # defined on SavingsAccount

`SavingsAccount` is-a `Account`: every operation defined on `Account` is available on `SavingsAccount`, and additional operations have been introduced. Python supports multiple inheritance (a class with more than one base), but in introductory use single inheritance is sufficient.

## 12. `super()` and cooperative initialization

`super()` returns a proxy object that delegates method calls to the parent class. Its most frequent use is in `__init__`, to make sure the parent's initialization runs before the subclass adds its own state.

In [ ]:
class SavingsAccount(Account):
    def __init__(
        self,
        owner: str,
        balance: float = 0.0,
        interest_rate: float = 0.02,
    ) -> None:
        super().__init__(owner, balance)   # delegate to Account.__init__
        self.interest_rate = interest_rate

Without the `super().__init__(...)` call, `Account.__init__` would never run, and the instance would lack `owner` and `balance`. `super()` also works inside other methods, not only `__init__`:

In [ ]:
class CheckingAccount(Account):
    def withdraw(self, amount: float) -> None:
        super().withdraw(amount)
        print(f"Withdrawal of {amount} processed")

In multiple-inheritance scenarios, `super()` follows the method resolution order (MRO) rather than walking up a single parent chain. The MRO is computed by the C3 linearization algorithm and can be inspected via `ClassName.__mro__`.

C3 produces a linearization (a flat ordering of classes) that satisfies three properties:

- A class appears before its parents. (Children override parents.)
- Parents appear in the order listed in the class definition. If you write class D(B, C), then B comes before C in the MRO.
- Monotonicity. If B precedes C in some class's MRO, then B precedes C in every MRO that contains both. No "reorderings" as you go down the hierarchy.

The third property is what makes C3 special — it's why Python rejects certain inheritance graphs as inconsistent rather than picking an arbitrary order. C3 will refuse to compute an MRO if no consistent order exists. The classic minimal example:

In [ ]:
class X: pass
class Y: pass
class A(X, Y): pass
class B(Y, X): pass
class C(A, B): pass   # TypeError: Cannot create a consistent MRO

## 13. Method overriding and polymorphism

A subclass can replace an inherited method by defining a method with the same name. The new implementation is used whenever the method is called on an instance of the subclass, regardless of whether the calling code knows the exact type.

In [ ]:
class CheckingAccount(Account):
    def __init__(
        self,
        owner: str,
        balance: float = 0.0,
        overdraft_limit: float = 0.0,
    ) -> None:
        super().__init__(owner, balance)
        self.overdraft_limit = overdraft_limit

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance + self.overdraft_limit:
            raise ValueError("Exceeds overdraft limit")
        self.balance -= amount


def process_payroll(accounts: list[Account], amount: float) -> None:
    for account in accounts:
        account.deposit(amount)


def collect_fees(accounts: list[Account], fee: float) -> None:
    for account in accounts:
        account.withdraw(fee)

`collect_fees` does not need to know which kind of account it has received. The correct `withdraw` implementation is selected based on the runtime type of each instance.

## 14. `isinstance` and `issubclass`

Type checks should be written with `isinstance` (for instances) and `issubclass` (for classes), not by comparing types directly. Direct comparison ignores subclasses.

In [ ]:
isinstance(savings, Account)           # True
isinstance(savings, SavingsAccount)    # True
isinstance(savings, CheckingAccount)   # False

issubclass(SavingsAccount, Account)    # True
issubclass(Account, SavingsAccount)    # False

# Tuple form: True if x is any of the given types
isinstance(value, (int, float))

Using `type(x) == Account` would return `False` for a `SavingsAccount` instance, which is rarely the intended behavior.

## 15. Composition vs. inheritance

Inheritance expresses an _is-a_ relationship: a `SavingsAccount` _is-a_ `Account`. Composition expresses a _has-a_ relationship: an `Account` _has-a_ transaction log, _has-a_ owner record, _has-a_ currency configuration. In Python, composition is often the better default because it keeps the class hierarchy shallow and avoids the fragile-base-class problem.

In [ ]:
class TransactionLog:
    def __init__(self) -> None:
        self.entries: list[tuple[str, float]] = []

    def record(self, kind: str, amount: float) -> None:
        self.entries.append((kind, amount))


class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.log = TransactionLog()

    def deposit(self, amount: float) -> None:
        self.balance += amount
        self.log.record("deposit", amount)

    def withdraw(self, amount: float) -> None:
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount
        self.log.record("withdraw", amount)

`Account` does not inherit from `TransactionLog`; it holds one. Replacing the log implementation later requires changing only the `Account` constructor, not a class hierarchy.

A practical guideline: reach for inheritance when the subclass is genuinely a specialized variant of the base class and shares its substitutability contract; reach for composition in most other cases.

## 16. Duck typing

Python relies heavily on duck typing: an object is treated as a particular kind of thing if it supports the operations that kind of thing requires, regardless of its declared type. The standard expression of the principle is, "If it walks like a duck and quacks like a duck, it is a duck."

In [ ]:
def total_balance(accounts):
    return sum(account.balance for account in accounts)

`total_balance` does not check that each item is an `Account`. Anything with a `balance` attribute will work — including instances of unrelated classes, mock objects in tests, or simple `SimpleNamespace` values. This is in deliberate contrast to languages where the parameter type would have to be declared and enforced.

The trade-off is that errors surface at runtime rather than at compile time. Static type checkers (`mypy`, `pyright`) and protocols (PEP 544) recover some of the benefits without giving up duck typing's flexibility.

## 17. Abstract base classes

When a hierarchy needs to declare that subclasses _must_ implement certain methods, the `abc` module provides abstract base classes. An abstract method is declared with `@abstractmethod`; instantiating a class that has unimplemented abstract methods raises `TypeError`.

In [ ]:
from abc import ABC, abstractmethod


class Account(ABC):
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    @abstractmethod
    def withdraw(self, amount: float) -> None:
        ...

    def deposit(self, amount: float) -> None:
        self.balance += amount


class CheckingAccount(Account):
    def withdraw(self, amount: float) -> None:
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


Account("Anna Smith")           # TypeError: cannot instantiate abstract class
CheckingAccount("Anna Smith")   # works

Abstract base classes also play a role in the standard library: `collections.abc` defines abstract types like `Iterable`, `Sized`, `Mapping`, which make `isinstance` checks meaningful for duck-typed protocols.

## 18. Special methods (dunders)

Methods whose names begin and end with double underscores hook into Python's syntax and built-in functions. Defining them lets a class behave like a built-in type.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def __repr__(self) -> str:
        return f"Account(owner={self.owner!r}, balance={self.balance!r})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return self.owner == other.owner and self.balance == other.balance

    def __hash__(self) -> int:
        return hash((self.owner, self.balance))

    def __lt__(self, other: "Account") -> bool:
        return self.balance < other.balance

    def __add__(self, other: "Account") -> "Account":
        if self.owner != other.owner:
            raise ValueError("Cannot merge accounts with different owners")
        return Account(self.owner, self.balance + other.balance)


a = Account("Anna Smith", 500.0)
b = Account("Anna Smith", 300.0)

a == b                  # __eq__
a < b                   # __lt__
a + b                   # __add__
sorted([b, a])          # __lt__ used by sorted
{a, b}                  # __hash__ used by sets and dicts
repr(a)                 # __repr__

Common groups of dunders include:

- Construction and representation: `__new__`, `__init__`, `__del__`, `__repr__`, `__str__`, `__format__`.
- Comparison: `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, `__hash__`.
- Arithmetic: `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__`, plus reflected (`__radd__`) and in-place (`__iadd__`) variants.
- Container behavior: `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__`, `__next__`.
- Callable and context: `__call__`, `__enter__`, `__exit__`.
- Truthiness: `__bool__`.

## 19. `__str__` vs. `__repr__`

`__repr__` is intended for developers; ideally it returns a string that unambiguously identifies the object — often something that could be evaluated to reconstruct it. `__str__` is intended for end-user output and is allowed to be more readable and less precise.

In [ ]:
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    def __repr__(self) -> str:
        return f"Account(owner={self.owner!r}, balance={self.balance!r})"

    def __str__(self) -> str:
        return f"{self.owner}: {self.balance:.2f} EUR"


account = Account("Anna Smith", 500.0)
repr(account)    # "Account(owner='Anna Smith', balance=500.0)"
str(account)     # "Anna Smith: 500.00 EUR"
print(account)   # uses __str__
account          # at the REPL: uses __repr__

If only `__repr__` is defined, `str(x)` falls back to it. The reverse does not hold. As a rule of thumb, every class worth printing should define `__repr__`; `__str__` is added when end-user formatting is needed.

## 20. Equality and hashing

By default, two instances compare equal only if they are the same object (`a is b`). Defining `__eq__` changes this. Once `__eq__` is overridden, `__hash__` should also be considered: an object is hashable if and only if it has a stable hash, and Python requires that objects which compare equal have the same hash.

In [ ]:
class Account:
    def __init__(self, account_id: str, owner: str, balance: float = 0.0) -> None:
        self.account_id = account_id
        self.owner = owner
        self.balance = balance

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return self.account_id == other.account_id

    def __hash__(self) -> int:
        return hash(self.account_id)

Python automatically sets `__hash__` to `None` when `__eq__` is defined without `__hash__`, making the instance unhashable. If instances should be usable as dictionary keys or set members, `__hash__` must be defined explicitly. Mutable objects are generally left unhashable, because changing the fields involved in hashing would corrupt any container holding the object.

## 21. Custom exceptions through inheritance

Inheritance is the standard mechanism for defining custom exception types. New exceptions inherit from `Exception` (or a more specific built-in exception class). Hierarchies of exceptions allow callers to catch families of related errors without listing each one.

In [ ]:
class AccountError(Exception):
    """Base class for all account-related errors."""


class InsufficientFundsError(AccountError):
    def __init__(self, requested: float, available: float) -> None:
        super().__init__(
            f"Requested {requested}, only {available} available"
        )
        self.requested = requested
        self.available = available


class AccountClosedError(AccountError):
    pass


class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance
        self.closed = False

    def withdraw(self, amount: float) -> None:
        if self.closed:
            raise AccountClosedError("Account is closed")
        if amount > self.balance:
            raise InsufficientFundsError(amount, self.balance)
        self.balance -= amount


try:
    account.withdraw(10_000)
except AccountError as exc:
    # catches both InsufficientFundsError and AccountClosedError
    print(exc)

Custom exception classes carry typed information (`requested`, `available` above), which is more reliable than parsing exception messages.

## 22. Dataclasses

Many classes exist primarily to hold structured data. Writing `__init__`, `__repr__`, and `__eq__` by hand for these classes is repetitive. The `dataclasses` module generates them from type-annotated class bodies.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Account:
    owner: str
    balance: float = 0.0
    transactions: list[float] = field(default_factory=list)

    def deposit(self, amount: float) -> None:
        self.balance += amount
        self.transactions.append(amount)


account = Account("Anna Smith", 500.0)
print(account)
# Account(owner='Anna Smith', balance=500.0, transactions=[])

account == Account("Anna Smith", 500.0)   # True

`@dataclass` accepts options to control which methods are generated: `frozen=True` produces immutable instances (with `__hash__` automatically defined), `order=True` adds comparison methods, `slots=True` (Python 3.10+) generates `__slots__`. Dataclasses replace boilerplate, not behavior — methods are written normally, as in any other class.

## 23. `classmethod` and `staticmethod`

Both decorators change how a method receives its first argument.

A `@classmethod` receives the class itself (conventionally `cls`) instead of an instance. The most common use is alternative constructors — methods that produce instances from data in a non-default form.

In [ ]:
import json
from datetime import date


class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self.balance = balance

    @classmethod
    def from_dict(cls, data: dict) -> "Account":
        return cls(owner=data["owner"], balance=data["balance"])

    @classmethod
    def from_json(cls, payload: str) -> "Account":
        return cls.from_dict(json.loads(payload))

    @classmethod
    def empty(cls, owner: str) -> "Account":
        return cls(owner=owner, balance=0.0)


account = Account.from_json('{"owner": "Anna Smith", "balance": 500.0}')
empty_account = Account.empty("Ben Jones")

Because `cls` is the actual class on which the method was called, `from_dict` produces instances of subclasses correctly: `SavingsAccount.from_dict({...})` returns a `SavingsAccount`.

A `@staticmethod` takes no implicit first argument; it is a plain function placed inside a class for namespacing. It is appropriate when a function is logically related to the class but does not depend on either the instance or the class.

In [ ]:
class Account:
    @staticmethod
    def is_valid_iban(iban: str) -> bool:
        cleaned = iban.replace(" ", "")
        return len(cleaned) >= 15 and cleaned[:2].isalpha()


Account.is_valid_iban("DE89 3704 0044 0532 0130 00")

If a method uses neither `self` nor `cls`, it is a candidate for `@staticmethod`. If it uses the class but not the instance, it is a candidate for `@classmethod`.

## 24. Classes are themselves objects

The principle introduced in Topic 2 — that everything is an object — applies to classes as well. A class can be assigned to a variable, stored in a container, passed to a function, or returned from one.

In [ ]:
account_types = [Account, SavingsAccount, CheckingAccount]

for cls in account_types:
    print(cls.__name__, cls.__mro__)


def make_account(account_class: type, owner: str, balance: float):
    return account_class(owner, balance)


account = make_account(SavingsAccount, "Anna Smith", 1000.0)

Every class is an instance of `type`. The `type` callable, when given three arguments, creates a new class dynamically:

In [ ]:
Account = type("Account", (), {"currency": "EUR"})

This is what `class Account: ...` ultimately does behind the scenes. Custom metaclasses — classes whose instances are themselves classes — are used to control class creation in libraries like ORMs and serialization frameworks. For ordinary application code, metaclasses are rarely needed; awareness that they exist is enough.

## 25. Real-world design principles

Once the mechanics of classes are clear, the practical question is when and how to use them. A handful of widely accepted guidelines apply to most situations.

**Prefer composition over inheritance.** Topic 15 introduced this trade-off in detail. Inheritance creates a tight, permanent coupling between subclass and base class: changes to the base class can affect every subclass, and the _is-a_ relationship is rarely as clean in practice as it appears on a whiteboard. Composition keeps the relationship explicit and replaceable. Inheritance is appropriate when a subclass genuinely substitutes for the base class in every context where the base class is used (the Liskov substitution principle); otherwise, composition is the safer default.

**Keep classes small.** A class that fits on a single screen is easier to read, test, and modify than one spanning several hundred lines. When a class grows beyond a small set of attributes and a few related methods, that growth is usually a signal that unrelated responsibilities have accumulated. Splitting the class into smaller pieces — typically composed together — almost always produces clearer code.

**One responsibility per class.** Each class should have a single reason to change. An `Account` class that handles balance arithmetic, persists itself to a database, sends email notifications, and formats reports has four reasons to change, and a change to any one of them risks breaking the others. The remedy is to separate concerns: `Account` for the domain logic, `AccountRepository` for persistence, `NotificationService` for email, `StatementFormatter` for reports. The classes then collaborate via composition.

**Avoid god objects.** A god object is a class that knows about and controls too much of the system. It typically arises gradually: a useful class accumulates "just one more" method, "just one more" attribute, until most of the program's logic lives in it. Symptoms include long parameter lists, dozens of methods, attributes that are used by only a subset of methods, and tests that require constructing the entire object graph in order to exercise a single behavior. The remedy is the same as for oversized classes: identify groups of related state and behavior and extract them into their own classes.

In [ ]:
# Before: one class doing too much
class Account:
    def __init__(self, owner: str, balance: float = 0.0) -> None: ...
    def deposit(self, amount: float) -> None: ...
    def withdraw(self, amount: float) -> None: ...
    def save_to_database(self) -> None: ...
    def load_from_database(self, account_id: str) -> None: ...
    def send_balance_email(self, recipient: str) -> None: ...
    def format_monthly_statement(self) -> str: ...
    def export_to_csv(self, path: str) -> None: ...


# After: responsibilities separated
class Account:
    def deposit(self, amount: float) -> None: ...
    def withdraw(self, amount: float) -> None: ...


class AccountRepository:
    def save(self, account: Account) -> None: ...
    def load(self, account_id: str) -> Account: ...


class StatementFormatter:
    def monthly(self, account: Account) -> str: ...
    def csv(self, account: Account, path: str) -> None: ...


class NotificationService:
    def send_balance(self, account: Account, recipient: str) -> None: ...

Each class is now responsible for one concern, and each can be tested or replaced in isolation.

## 26. Common mistakes

A short collection of errors that are easy to make and worth recognizing early.

**Mutable class attributes shared between instances.** A list or dictionary defined at class level is shared across all instances unless explicitly replaced. This is the issue introduced in Topic 7; it is among the most common Python class bugs.

In [ ]:
# Wrong
class Account:
    transactions = []

    def deposit(self, amount: float) -> None:
        self.transactions.append(amount)


a = Account()
b = Account()
a.deposit(50)
b.transactions   # [50] — shared with a

The fix is to assign the mutable object inside `__init__`, so each instance gets its own:

In [ ]:
# Correct
class Account:
    def __init__(self) -> None:
        self.transactions: list[float] = []

**Forgetting `self`.** Methods defined inside a class must declare `self` as their first parameter, and instance attributes must be accessed through it.

In [ ]:
# Wrong
class Account:
    def deposit(amount: float) -> None:        # missing self
        self.balance += amount                   # NameError when called


# Wrong
class Account:
    def __init__(self, balance: float) -> None:
        self.balance = balance

    def deposit(self, amount: float) -> None:
        balance += amount                        # local variable, not the attribute


# Correct
class Account:
    def __init__(self, balance: float) -> None:
        self.balance = balance

    def deposit(self, amount: float) -> None:
        self.balance += amount

The first error produces a `TypeError` (the instance is passed as the first argument but the method expects `amount` to be it). The second is more insidious: it runs without an exception but silently fails to update the instance.

**Abusing inheritance.** Inheritance is sometimes used to reuse code rather than to express an _is-a_ relationship. The result is a hierarchy that does not reflect any real conceptual structure, and one that becomes harder to change as the codebase grows.

In [ ]:
# Wrong: an Account is not an EmailSender
class EmailSender:
    def send(self, recipient: str, message: str) -> None: ...


class Account(EmailSender):     # inherits send() "for free"
    def deposit(self, amount: float) -> None: ...


# Correct: compose the dependency in
class EmailSender:
    def send(self, recipient: str, message: str) -> None: ...


class Account:
    def __init__(self, email_sender: EmailSender) -> None:
        self.email_sender = email_sender

    def deposit(self, amount: float) -> None: ...

The test is whether the _is-a_ relationship would read naturally to a domain expert. "An account is an email sender" does not. If two classes share code but neither is a specialization of the other, composition or a free-standing helper function is the right answer.

**Comparing with `type(x) ==`.** Direct comparison of types ignores subclasses, which is rarely what is intended.

In [ ]:
# Wrong
if type(account) == Account:
    ...   # False for SavingsAccount, CheckingAccount, and any other subclass


# Correct
if isinstance(account, Account):
    ...   # True for Account and any subclass

The exception is when the _exact_ type matters and subclass instances must be excluded — for example, when dispatching on type in a serialization layer. This is uncommon; when it occurs, a comment explaining the intent is worth adding, since most readers will assume `isinstance` was meant.